In [27]:
import os
import time
from openai import OpenAI
from tqdm.auto import tqdm

In [ ]:
MODEL = "gpt-5-mini"

INPUT_BASE = "data/chunked_train_data"
OUTPUT_BASE = "results/openai_results/openai_summaries_train_chunked"

DELAY_SECONDS = 1
MAX_RETRIES = 3

client = OpenAI(api_key=OPEN_API_KEY)

### System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a legal research assistant specialized in summarizing legislative documents.
The included text is part of a larger summary.

Create accurate, concise, and short summaries in plain English.
Preserve key legal clauses, definitions, obligations, and relationships.
Do not introduce new information.
Avoid unnecessary simplification.
Maintain legal terminology where important.
"""

### Summarization Function

In [ ]:
def summarize_text(text):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL,
                max_output_tokens=1024,
                input=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": f"Summarize the following legal text:\n\n{text}"
                    }
                ]
            )

            return response.output_text  

        except Exception as e:
            print(f"Retry {attempt+1} due to error: {e}")
            time.sleep(2)

    return None

### Processing Fucntion

In [31]:
def process_train_chunks():
    os.makedirs(OUTPUT_BASE, exist_ok=True)

    # List all .txt chunk files
    files = [f for f in os.listdir(INPUT_BASE) if f.endswith(".txt")]

    for filename in tqdm(files, desc="Processing train chunks"):

        input_path = os.path.join(INPUT_BASE, filename)
        output_path = os.path.join(OUTPUT_BASE, filename)

        # Skip if already processed
        if os.path.exists(output_path):
            continue

        with open(input_path, "r", encoding="utf-8") as f:
            text = f.read()

        summary = summarize_text(text)

        if summary:
            # Save immediately as .txt
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(summary)

        time.sleep(DELAY_SECONDS)

    print("All train summaries completed.")

In [32]:
process_train_chunks()

Processing train chunks: 100%|██████████| 6021/6021 [23:39:45<00:00, 14.15s/it]   

All train summaries completed.
